# Pipeline de Benchmark do TCC

Roda o sistema de benchmark e documenta o automador completo do TCC:
download/preparação do dataset, split estratificado, treino, inferência,
tabelas LaTeX, figuras PNG, `dataset.md` e `tcc_report.md`.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

_REPO_URL = "https://github.com/thierrybraga/XFakeSong.git"


def _project_root() -> Path:
    p = Path.cwd()
    while not (p / "app").exists() and p.parent != p:
        p = p.parent
    return p


# Colab/Kaggle limpos: clona o repositório se o projeto não estiver no disco.
if not (_project_root() / "app").exists():
    if not Path("XFakeSong").exists():
        print("Clonando XFakeSong...")
        subprocess.run(["git", "clone", "--depth", "1", _REPO_URL], check=True)
    os.chdir("XFakeSong")

_DEPS = {"librosa": "librosa>=0.10", "soundfile": "soundfile>=0.12",
         "pywt": "PyWavelets>=1.4"}
_missing = [s for m, s in _DEPS.items() if importlib.util.find_spec(m) is None]
if _missing:
    print("Instalando dependências de áudio:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT, "| Colab:", "google.colab" in sys.modules)

## 1. Benchmark rápido (sintético) — valida o harness em segundos

Treina algumas arquiteturas em **dados sintéticos** e gera as MESMAS
métricas, tabela e figuras do relatório do TCC. Como os dados são
sintéticos, **os números não são resultados reais** (acurácia ~50% é
esperada) — servem só para validar o harness de ponta a ponta. Seed fixo
(`42`, default do benchmark). Resultados reais: Seção 2.

In [ ]:
import pandas as pd

try:
    from IPython.display import Image, display
except Exception:           # fora do kernel IPython
    display, Image = print, None

from benchmarks import BenchmarkConfig, run_benchmark

try:
    import tensorflow as tf
    print("GPUs disponíveis:", len(tf.config.list_physical_devices("GPU")))
except Exception:
    pass

OUT = ROOT / "results" / "notebook_benchmark"
cfg = BenchmarkConfig.quick(
    architectures=["MultiscaleCNN", "Ensemble", "SVM", "RandomForest"],
    synthetic_n=160,
    snr_levels_db=[20],
    output_dir=str(OUT),
)
run_benchmark(cfg)

# Tabela canônica do relatório, ordenada por EER (menor = melhor).
cols = ["arquitetura", "convergiu", "accuracy", "eer", "auc_roc",
        "min_tdcf", "params", "size_mb", "latency_ms"]
df = pd.read_csv(OUT / "results.csv")
df = df[[c for c in cols if c in df.columns]].sort_values("eer")
display(df.round(4))

### Figuras agregadas geradas pelo benchmark

In [ ]:
# Exibe inline as figuras que o benchmark gravou em disco.
figs = OUT / "figures"
for name in ["roc.png", "score_distributions.png", "confusion_matrices.png",
             "eficiencia.png", "robustez.png"]:
    fp = figs / name
    if fp.exists() and Image is not None:
        print(name)
        display(Image(filename=str(fp)))
    elif fp.exists():
        print("figura:", fp)

## 2. Execução completa do TCC

> ⚠️ **Pesado — requer GPU e tempo.** Baixa/prepara ~20k amostras (10k
> real + 10k fake) e treina o preset completo de arquiteturas. Em GPU leva
> de dezenas de minutos a horas; em CPU é inviável. Por isso fica
> **desligado por padrão** (`RUN_FULL_PIPELINE = False`). O
> `--tcc-full-dataset` já ativa download + benchmark completo + probe da API.

A execução completa grava o relatório em `results/tcc_full_20k/`:
`dataset.md`, `dataset_manifest.json`, `results.json`/`results.csv`,
`tcc_report.md` e as figuras `figures/roc.png`,
`figures/confusion_matrices.png` e `figures/score_distributions.png`
(estes dois primeiros, `dataset.md`/`dataset_manifest.json`, só saem aqui —
a Seção 1 não baixa dataset).

In [ ]:
import subprocess

RUN_FULL_PIPELINE = False
OUTPUT_DIR = ROOT / "results" / "tcc_full_20k"
DATASET_NPZ = ROOT / "app" / "datasets" / "benchmark_audio_raw_20k.npz"

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "run_tcc_pipeline.py"),
    "--tcc-full-dataset",
    "--out", str(OUTPUT_DIR),
    "--npz", str(DATASET_NPZ),
]
print(" ".join(cmd))

if RUN_FULL_PIPELINE:
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("Execução completa desativada. Defina RUN_FULL_PIPELINE = True para rodar.")

Comando equivalente no terminal:

```bash
python scripts/run_tcc_pipeline.py --tcc-full-dataset \
    --out results/tcc_full_20k \
    --npz app/datasets/benchmark_audio_raw_20k.npz
```

Ou o benchmark direto sobre um `.npz` já exportado (veja como gerar o
dataset em `docs/12_DATASETS.md`):

```bash
python scripts/run_benchmark.py --full --dataset app/datasets/SEU_DATASET.npz
```

Veja `docs/15_BENCHMARK.md` para o mapeamento saída → tabela/figura do TCC.

## 3. Validar e ler artefatos do relatório

In [ ]:
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# Lê o relatório que existir: o completo (Seção 2) OU o rápido (Seção 1) —
# assim a tabela aparece mesmo sem rodar o pipeline de horas.
candidates = [ROOT / "results" / "tcc_full_20k",
              ROOT / "results" / "notebook_benchmark"]
report_dir = next((d for d in candidates if (d / "results.csv").exists()),
                  candidates[0])
print("Lendo relatório de:", report_dir)

# dataset.md/manifest só existem na execução completa (Seção 2).
required = ["results.csv", "results.json", "tcc_report.md",
            "figures/roc.png", "figures/confusion_matrices.png",
            "figures/score_distributions.png"]
missing = [p for p in required if not (report_dir / p).exists()]
print("Ausentes:", missing or "nenhum")

if (report_dir / "results.csv").exists():
    display(pd.read_csv(report_dir / "results.csv"))